# 03 · LLM fine-tuning + Propose-Verify-Repair (GPU)
Add data: both ROI shards and the output of notebook 02 (for `best.pt`). Accelerator **GPU T4**.

Llama 3.2 is gated: accept its licence on huggingface.co and add a Kaggle secret **HF_TOKEN**. Without it, set `LLM` to an open model such as `Qwen/Qwen2.5-1.5B`.

In [ ]:
# --- setup: clone the project code (private repo: add a Kaggle secret GITHUB_TOKEN) ---
import os, subprocess
url = "https://github.com/durgaanuhya02/MoVi-Swin-Motion--and-Viseme-Aware-Space-Time-Swin-Transformer-for-Robust-and-Efficient-Lip-Reading.git"
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = url.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", url, "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
!pip install -q cmudict peft bitsandbytes

In [ ]:
import glob, os
INDEX = sorted(glob.glob('/kaggle/input/**/grid_rois_shard*/index.jsonl', recursive=True))
VISUAL = glob.glob('/kaggle/input/**/visual_grid/best.pt', recursive=True)[0]
LLM = 'meta-llama/Llama-3.2-1B'
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret:', e)
INDEX_ARGS = ' '.join(INDEX)
print(INDEX, VISUAL)

## Stage 2 · phoneme→sentence corpus and LoRA fine-tuning

In [ ]:
!cat {INDEX_ARGS} > /kaggle/working/grid_index_all.jsonl
!python -m src.llm.build_corpus --manifest /kaggle/working/grid_index_all.jsonl --split train --wikitext 50000 --out /kaggle/working/llm_corpus.jsonl

In [ ]:
!python -m src.llm.train_p2t --config configs/llm_p2t.yaml --set model_name={LLM} max_steps=3000

## Stage 3 · tune λ on validation, then evaluate on the unseen-speaker test set

In [ ]:
ADAPTER = '/kaggle/working/runs/llm_p2t/adapter'
!python -m src.pvr.run --visual {VISUAL} --index {INDEX_ARGS} --split val --llm {LLM} --adapter {ADAPTER} --out /kaggle/working/runs/pvr_val --n 5 --limit 300 --sweep | tail -40

In [ ]:
import json
sweep = json.load(open('/kaggle/working/runs/pvr_val/metrics.json'))['lambda_sweep']
LAM = float(min(sweep, key=sweep.get)); print('best lambda', LAM, sweep)

In [ ]:
!python -m src.pvr.run --visual {VISUAL} --index {INDEX_ARGS} --split test --llm {LLM} --adapter {ADAPTER} --out /kaggle/working/runs/pvr_test --n 5 --lam {LAM} --sweep | tail -40